In [1]:
# 📦 Required installs
# !pip install langchain openai faiss-cpu
#!pip install langchain-core

In [2]:


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")


# 1. Load the text file
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 2. Split the text into chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
documents = [Document(page_content=chunk) for chunk in texts]

# 3. Create vector store
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")
vectorstore = FAISS.from_documents(documents, embedding=embedding)

# 4. Setup LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", temperature=0)


C:\Users\user\AppData\Local\Temp\ipykernel_8232\1253199841.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
#!pip show langchain-community
#!pip install langchain-classic

In [4]:
#Get retrieves --> it does not generate
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor using LLM
compressor = LLMChainExtractor.from_llm(llm)

# Create compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# Run retrieval
print("🔹 Contextual Compression Results:")
results = compression_retriever.invoke("Who created LangChain?")
for doc in results:
    print("-", doc.page_content)


🔹 Contextual Compression Results:


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


- LangChain was created by Harrison Chase.


In [5]:
#Integration with ConversationalRetrievalChain- Without agent mode
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory


# ✅ ContextualCompressionRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectorstore.as_retriever()
)

# ✅ Memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

# ✅ Chain
rag_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=compression_retriever, memory=memory)

# 🧪 Ask a few questions
print("🔹 ConversationalRetrievalChain:")
print(rag_chain.invoke({"question": "What is LangChain?"})["answer"])
#print(rag_chain.invoke({"question": "Who created it?"})["answer"])


C:\Users\user\AppData\Local\Temp\ipykernel_8232\3191196239.py:17: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.valida

🔹 ConversationalRetrievalChain:


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


LangChain is a framework for building applications with LLMs.


In [6]:
#Integration into Agent (with Tool)

from langchain_classic.agents import initialize_agent, AgentType
from langchain_core.tools import StructuredTool

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": [memory.chat_memory.messages ]
    })["answer"]

# Structured Tool
rag_tool = StructuredTool.from_function(
    name="RAG_Tool",
    description="Answer LangChain-related questions with context.",
    func=rag_tool_fn
)

# Agent with memory
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

# 🧪 Ask via agent
print("\n🔹 Agent Conversation:")
print(agent.run("What is LangChain?"))
#print(agent.run("Who created it?"))


C:\Users\user\AppData\Local\Temp\ipykernel_8232\1335081987.py:21: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(
C:\Users\user\AppData\Local\Temp\ipykernel_8232\1335081987.py:31: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  print(agent.run("What is LangChain?"))
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🔹 Agent Conversation:


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: RAG_Tool
Action Input: What is LangChain?

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleGenerativeAIError: Error embedding content: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}

In [7]:
#multiQuery
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.chains import RetrievalQA


# MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)

# RetrievalQA Chain
rag_multi = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=multi_query_retriever,
    return_source_documents=True
)

# 🧪 Ask question
#print("\n🔹 RAG Pipeline (MultiQueryRetriever):")
#res = rag_multi.run("Tell me about LangChain creator and features.")
#print(res)

# 🧪 Ask question
print("\n🔹 RAG Pipeline (MultiQueryRetriever):")
res = rag_multi.invoke({"query": "Tell me about LangChain creator and features."})

print("Answer:", res["result"])
print("\nSources:")
for doc in res["source_documents"]:
    print("-", doc.page_content[:200])  # print first 200 chars of each doc




🔹 RAG Pipeline (MultiQueryRetriever):


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: Based on the provided context:

* **Creator:** LangChain was created by Harrison Chase.
* **Features:** LangChain supports RAG (Retrieval-Augmented Generation), agents, memory, tools, and more. It is commonly used in chatbots, document Q&A, and AI workflows.

Sources:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&
